# 08 - Final model and `submission.csv`

1. **Choose** the final model for each task using the validation set only (never the Kaggle
   leaderboard). Candidates, built from the validation predictions saved by notebooks 02, 04 and 05:
   the best k-NN, the best Random Forest, the 5-network ensemble, and averages of these.

   We don't use the single seed-42 network as a candidate: that run *is* the winner of notebook 05's
   search on this same validation set, so its score is inflated by the selection. The 5-network
   ensemble is at least as accurate and more stable.
2. **Retrain** the chosen model on **all** labelled rows (20,000 images / 12,000 tweets), with the
   preprocessing re-fitted on them. For networks the number of epochs is fixed to what validation chose,
   since there is no validation set left to stop on.
3. **Predict** the 10,000 test rows and write `submission.csv`, matched to `sample_submission.csv` by id.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch

from src.data import get_split, load_task, DATA_DIR, ROOT, N_CLASSES, SEED, SENTIMENT_NAMES
from src.preprocessing import Preprocessor
from src.knn import KNNClassifier
from src.random_forest import RandomForestClassifier
from src.neural_net import train_mlp, predict_proba
from src.utils import accuracy, save_table, load_predictions, set_seed

set_seed()
torch.set_num_threads(1)

## 1. Choose the final model per task

In [2]:
CANDIDATES = {                     # name -> models whose probabilities are averaged
    "k-NN": ["knn"],
    "Random Forest": ["rf"],
    "NN ensemble (5 seeds)": ["nn5"],
    "NN ensemble + k-NN": ["nn5", "knn"],
    "NN ensemble + k-NN + RF": ["nn5", "knn", "rf"],
}
rows, choice = [], {}
for task in ("image", "sentiment"):
    y_va = get_split(task)[3]
    val_proba = {m: load_predictions(task, m)[1] for m in ("knn", "rf", "nn5")}
    scores = {}
    for name, members in CANDIDATES.items():
        p = np.mean([val_proba[m] for m in members], axis=0)
        scores[name] = accuracy(y_va, p.argmax(1))
        rows.append(dict(task=task, candidate=name, val_acc=scores[name]))
    choice[task] = max(scores, key=scores.get)
candidates = pd.DataFrame(rows)
save_table(candidates, "final_candidates")
print("chosen:", choice)
candidates.pivot(index="candidate", columns="task", values="val_acc").loc[list(CANDIDATES)].round(4)

chosen: {'image': 'k-NN', 'sentiment': 'NN ensemble (5 seeds)'}


task,image,sentiment
candidate,,
k-NN,0.6148,0.5933
Random Forest,0.3662,0.6029
NN ensemble (5 seeds),0.4078,0.6792
NN ensemble + k-NN,0.5365,0.6762
NN ensemble + k-NN + RF,0.5402,0.6667


* **Images: k-NN alone.** Averaging it with the network or forest makes it *worse*: they mostly get
  different images right, and their confident wrong answers outvote k-NN (see notebook 07).
* **Sentiment: the 5-network ensemble.**

## 2. Retrain on all labelled data and predict the test set

In [3]:
def train_and_predict(member, task, X, y, X_test):
    """Train one member on ALL labelled rows and return its test-set probabilities."""
    if member == "knn":
        cfg = load_predictions(task, "knn")[2]["config"]
        prep = Preprocessor(cfg["preprocessing"]).fit(X)
        knn = KNNClassifier(k=cfg["k"], metric=cfg["metric"], weights=cfg["weights"], power=cfg["power"],
                            n_classes=N_CLASSES[task]).fit(prep.transform(X), y)
        return knn.predict_proba(prep.transform(X_test))
    if member == "rf":
        cfg = load_predictions(task, "rf")[2]["config"]
        rf = RandomForestClassifier(random_state=SEED, n_jobs=4, **cfg).fit(X, y, N_CLASSES[task])
        return rf.predict_proba(X_test)
    if member == "nn5":
        info = load_predictions(task, "nn")[2]
        cfg = dict(info["config"], hidden=tuple(info["config"]["hidden"]))
        cfg["epochs"] = int(np.median(info["best_epochs_5_seeds"]))   # epochs chosen on validation
        prep = Preprocessor(info["preprocessing"]).fit(X)
        A, T = prep.transform(X), prep.transform(X_test)
        probs = []
        for s in range(5):
            model, _, _ = train_mlp(A, y, n_classes=N_CLASSES[task], seed=SEED + s, **cfg)
            probs.append(predict_proba(model, T))
        print(f"  {task}: 5 networks, {cfg}")
        return np.mean(probs, axis=0)


test_predictions = []
for task in ("image", "sentiment"):
    X, y, X_test, test_ids = load_task(task)
    p = np.mean([train_and_predict(m, task, X, y, X_test) for m in CANDIDATES[choice[task]]], axis=0)
    test_predictions.append(pd.DataFrame({"id": test_ids, "label": p.argmax(1)}))
    print(f"{task}: final model = {choice[task]}, trained on {len(y)} rows")

image: final model = k-NN, trained on 20000 rows


  sentiment: 5 networks, {'hidden': (512, 256), 'activation': 'relu', 'dropout': 0.5, 'lr': 0.001, 'optimizer': 'adam', 'weight_decay': 0.0001, 'batch_size': 128, 'epochs': 9}
sentiment: final model = NN ensemble (5 seeds), trained on 12000 rows


## 3. Build and check `submission.csv`
Merge on `id` with the template (its row order is different from the test files), then check it.

In [4]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
submission = sample[["id"]].merge(pd.concat(test_predictions, ignore_index=True), on="id",
                                  how="left", validate="one_to_one")
is_image = submission["id"].str.startswith("img_")
assert len(submission) == 10000 and submission["id"].is_unique
assert submission["label"].notna().all()
submission["label"] = submission["label"].astype(int)
assert is_image.sum() == 5000
assert submission.loc[is_image, "label"].between(0, 49).all()
assert submission.loc[~is_image, "label"].between(0, 2).all()
submission.to_csv(ROOT / "submission.csv", index=False)
print("saved submission.csv")
submission.head()

saved submission.csv


,id,label
0,text_003234,2
1,text_004517,1
2,img_000140,39
3,img_004880,15
4,text_002494,0


### Sanity check: predicted label counts on the test set

In [5]:
image_counts = submission.loc[is_image, "label"].value_counts()
print("image: predicted images per class: min %d, median %d, max %d"
      % (image_counts.min(), image_counts.median(), image_counts.max()))
sentiment_counts = submission.loc[~is_image, "label"].value_counts().sort_index()
print("sentiment:", {SENTIMENT_NAMES[k]: int(v) for k, v in sentiment_counts.items()})

image: predicted images per class: min 62, median 98, max 138
sentiment: {'Negative': 2342, 'Neutral': 1380, 'Positive': 1278}


* **Images:** the test set has exactly 100 images per class, and k-NN predicts each class between
  62 and 138 times. Some classes still attract more neighbours than others, but this is much more
  even than the cosine k-NN we started with (44 to 175 on the same test set).
* **Sentiment:** 47% of test tweets are predicted Negative (37% on the balanced validation set).
  Either the test set has more negative tweets than the balanced training set (TweetEval's original
  test split is not balanced), or the test tweets are a bit different. We can't check without labels,
  and we did not change anything based on the test set.